# Agenter Quickstart

This notebook demonstrates the core features of Agenter - the backend-agnostic SDK for autonomous coding agents.

## Setup

```bash
# Install core agenter with all backends and adapters used in this notebook
pip install agenter[codex,claude-code,langgraph,pydantic-ai]

# Set the API key for your LLM provider
export ANTHROPIC_API_KEY="your-api-key"  # Anthropic
export OPENAI_API_KEY="your-api-key"     # OpenAI

# Pick your backend (defaults to "anthropic-sdk")
export ACA_DEFAULT_BACKEND="codex"  # Options: anthropic-sdk, claude-code, codex, openhands
```

In [1]:
import tempfile
from pathlib import Path

from agenter import AutonomousCodingAgent, CodingRequest, configure_logging

# Silence verbose debug logs for cleaner notebook output
configure_logging(level="WARNING")

# Use configure_logging(level="DEBUG") to see all internal logs

## 1. Basic Usage

The simplest way to use Agenter: give it a prompt and a directory.

In [2]:
# Create a temporary workspace
workspace_1 = tempfile.mkdtemp(prefix="agenter_demo_")

agent = AutonomousCodingAgent()
result = await agent.execute(
    CodingRequest(
        prompt="Create a minimal Python function that calculates Fibonacci numbers.",
        cwd=workspace_1,
    )
)
result

**Status:** completed

**Summary:** Task completed successfully


**fibonacci.py**
```py
def fib(n: int) -> int:
    """Return the nth Fibonacci number for n >= 0."""
    if n < 0:
        raise ValueError("n must be non-negative")
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

```

## 2. Streaming Events

**Built-in display**: Use `verbosity=Verbosity.NORMAL` for Rich-formatted output.

**Custom event handling**: Process events exactly how you want (for example, for your custom UI or logging setup).

In [3]:
from agenter import Verbosity
from agenter.data_models import CodingEventType
from agenter.data_models.messages import ToolCallMessage

workspace_2 = tempfile.mkdtemp(prefix="agenter_stream_")

agent = AutonomousCodingAgent()
request = CodingRequest(
    prompt="Create a calculator module with add, subtract, multiply, divide.",
    cwd=workspace_2,
)

tools_used = []
async for event in agent.stream_execute(request, verbosity=Verbosity.NORMAL):
    if event.type == CodingEventType.BACKEND_MESSAGE and isinstance(event.message, ToolCallMessage):
        tools_used.append(event.message.tool_name)

print(f"List of tool used: {tools_used}")

╔════════════════════════════════════════════════════ Agenter ════════════════════════════════════════════════════╗
║ Working directory: /var/folders/0f/1wtyzkj508nbh_hny1k3rrbm0000gn/T/agenter_stream__bqju_22                     ║
║ Model: gpt-5.1-codex                                                                                            ║
╚═════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝

▶ Iteration 1

╭─────────────────────────────────────────────────── 📝 Prompt ───────────────────────────────────────────────────╮
│ Create a calculator module with add, subtract, multiply, divide.                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 📨 Response (0 tokens) ─────────────────────────────────────────────╮
│ - Added `calculator.py:1` defining `add`, `subtract`, `multiply`, and `divide`; simple docstrings and type      │
│ hints keep the API clear, and `divide` guards against a zero denominator with an explicit error.                │
│                                                                                                                 │
│ Next steps (optional):                                                                                          │
│ 1) Add unit tests (e.g., `pytest`) to lock behavior across edge cases.                                          │
│ 2) Extend the module with higher-level helpers (percentages, exponentiation) if needed.                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── ✓ Validation ──────────────────────────────────────────────────╮
│ All checks passed                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╔════════════════════════════════════════════════════ Summary ════════════════════════════════════════════════════╗
║   Status:            ✅ Completed                                                                               ║
║   Iterations:        1                                                                                          ║
║   Tokens:            112                                                                                        ║
║   Duration:          11.9s                                                                                      ║
║   Files modified:    1                                                                                          ║
║                        • calculator.py                                                                          ║
╚═════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝

List of tool used: []


## 3. Structured Outputs

Get typed, validated outputs using Pydantic models.

In [4]:
from pydantic import BaseModel


class CodeAnalysis(BaseModel):
    """Structured output for code analysis."""

    summary: str
    issues: list[str]
    suggestions: list[str]
    complexity_score: int  # 1-10


workspace_3 = tempfile.mkdtemp(prefix="agenter_structured_")

# Create a file to analyze
Path(workspace_3, "example.py").write_text("""
def process_data(data):
    result = []
    for item in data:
        if item > 0:
            for i in range(item):
                if i % 2 == 0:
                    result.append(i * item)
    return result
""")

agent = AutonomousCodingAgent()
result = await agent.execute(
    CodingRequest(
        prompt="Analyze the code in example.py for complexity and potential issues.",
        cwd=workspace_3,
        output_type=CodeAnalysis,  # With structured output
    )
)

analysis: CodeAnalysis = result.output

print(f"Summary: {analysis.summary}")
print(f"Complexity: {analysis.complexity_score}/10")
print(f"Issues: {analysis.issues}")
print(f"Suggestions: {analysis.suggestions}")

Summary: `process_data` uses nested loops and modulo checks to build a list of even-index multiples for every positive value, but it has no documentation and silently skips non‑positive inputs.
Complexity: 6/10
Issues: ['Complexity grows with the sum of positive input magnitudes (O(Σ item)), which can become large for big values.', 'The function name implies general processing, yet it silently drops non-positive numbers without warning or logging.', 'No docstring or type hints explain expected input types or numerical ranges, making misuse likelier.', 'Appending every intermediate value can exhaust memory when `item` values are large.']
Suggestions: ['Document expected input (iterable of positive ints) and clarify behavior for zero/negative values, or explicitly handle them.', 'Consider yielding results lazily (generator) or capping iterations to reduce both time and memory costs for large numbers.', 'Add basic validation or logging for unexpected inputs so callers notice when values a

## 4. Switching Backends

Switch between backends without changing your code.

In [5]:
workspace_4 = tempfile.mkdtemp(prefix="agenter_backends_")

task = CodingRequest(
    prompt="Create a hello.py that prints 'Hello, World!'",
    cwd=workspace_4,
)

# Same task, different backends - each uses its own default model
for backend in ["anthropic-sdk", "codex"]:
    agent = AutonomousCodingAgent(backend=backend)
    result = await agent.execute(task)
    print(f"{backend}: {result.status} (${result.total_cost_usd:.4f})")

anthropic-sdk: CodingStatus.COMPLETED ($0.0086)


codex: CodingStatus.COMPLETED ($0.0003)


## 5. Framework Adapters

Adapters **subclass framework base classes** to set up tracing and all framework features automatically.

- **LangGraph**: `create_coding_node()` returns `RunnableLambda` → LangSmith tracing works when `LANGSMITH_API_KEY` is set
- **PydanticAI**: `CodingAgent` subclasses `pydantic_ai.Agent` → Logfire tracing works automatically

In [6]:
# PydanticAI adapter - CodingAgent IS-A pydantic_ai.Agent
from agenter.adapters.pydantic_ai import CodingAgent

workspace_5 = tempfile.mkdtemp(prefix="agenter_pydantic_")

# CodingAgent subclasses pydantic_ai.Agent and overrides .run()
# Using codex backend (OpenAI models)
agent = CodingAgent(cwd=workspace_5, backend="codex")

result = await agent.run("Create a hello world script.")

print(f"Status: {result.status}")
print(f"Summary: {result.summary}")

Status: CodingStatus.COMPLETED
Summary: Task completed successfully


In [7]:
# LangGraph adapter - returns RunnableLambda for LangSmith tracing
from agenter.adapters.langgraph import create_coding_node

workspace_6 = tempfile.mkdtemp(prefix="agenter_langgraph_")

# create_coding_node returns RunnableLambda (langchain base class)
# Using anthropic-sdk
coding_node = create_coding_node(cwd=workspace_6, backend="anthropic-sdk")

# Use ainvoke() - LangSmith traces automatically
state = {"prompt": "Create a utility function to format dates."}
result_state = await coding_node.ainvoke(state)

print(f"Status: {result_state['coding_result']['status']}")
print(f"Summary: {result_state['coding_result']['summary']}")

Status: CodingStatus.COMPLETED
Summary: Task completed successfully


## Cleanup

In [8]:
import shutil

for ws in [workspace_1, workspace_2, workspace_3, workspace_4, workspace_5, workspace_6]:
    try:
        shutil.rmtree(ws)
        print(f"Dir '{ws}' cleaned up.")
    except Exception:
        pass

Dir '/var/folders/0f/1wtyzkj508nbh_hny1k3rrbm0000gn/T/agenter_demo_grommogf' cleaned up.
Dir '/var/folders/0f/1wtyzkj508nbh_hny1k3rrbm0000gn/T/agenter_stream__bqju_22' cleaned up.
Dir '/var/folders/0f/1wtyzkj508nbh_hny1k3rrbm0000gn/T/agenter_structured_ut5pripw' cleaned up.
Dir '/var/folders/0f/1wtyzkj508nbh_hny1k3rrbm0000gn/T/agenter_backends_mjy61qyq' cleaned up.
Dir '/var/folders/0f/1wtyzkj508nbh_hny1k3rrbm0000gn/T/agenter_pydantic_lrt8rvp_' cleaned up.
Dir '/var/folders/0f/1wtyzkj508nbh_hny1k3rrbm0000gn/T/agenter_langgraph_a4lzgsgm' cleaned up.
